# GA Search for BRAIN Alphas

This notebook implements a self-contained genetic algorithm (GA) search for alpha expressions on WorldQuant BRAIN. Fitness is a composite score derived from the IS Sharpe, fitness metric, and turnover returned by the BRAIN simulation API.

The workflow is:

1. **Authenticate**: start a `SingleSession` via `ace_lib`
2. **Fetch datafields**: pull available fields from the BRAIN API for the target region/universe
3. **Initialise population**: sample random individuals from template library
4. **Evaluate**: batch-simulate individuals and score them
5. **Evolve**: tournament selection → crossover → mutation for `N_GENERATIONS` generations
6. **Inspect results**: display the Hall of Fame and generation logbook

Credentials are read from `secrets/platform-brain.json`

## 1. Imports and session

depend on `ace_lib`, `deap`

In [ ]:
import json
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from deap import base, creator, tools

import ace_lib as ace
from ace_lib import SingleSession

with open('secrets/platform-brain.json') as _f:
    _creds = json.load(_f)

s = ace.start_session(_creds['email'], _creds['password'])

## 2. Search configuration

`REGION` and `UNIVERSE` control which datafields are fetched and which simulation settings are available

GA parameters:

- `POPULATION_SIZE`: number of individuals evaluated each generation.
- `N_GENERATIONS`: number of evolutionary generations
- `CXPB`: probability that a selected pair undergoes crossover
- `MUTPB`: probability that a selected individual is mutated
- `TOURNAMENT_SIZE`: number of individuals competing in each tournament selection
- `BATCH_SIZE`: number of alphas sent to `simulate_multi_alpha`

**Fitness thresholds**: individuals that fall below these are considered infeasible

In [ ]:
REGION     = 'USA'
UNIVERSE   = 'TOP3000'
DELAY      = 1

SIM_SETTINGS = {
    'instrumentType':  'EQUITY',
    'region':           REGION,
    'universe':         UNIVERSE,
    'delay':            DELAY,
    'truncation':       0.01,
    'pasteurization': 'ON',
    'testPeriod':      'P0Y0M0D',
    'unitHandling':    'VERIFY',
    'nanHandling':     'OFF',
    'maxTrade':        'OFF',
    'language':        'FASTEXPR',
    'visualization': False,
}

# GA settings
POPULATION_SIZE  = 20
N_GENERATIONS    = 10
CXPB             = 0.6
MUTPB             = 0.4
TOURNAMENT_SIZE  = 3
BATCH_SIZE       = 4
HOF_SIZE         = 5

# hard thresholds
MIN_SHARPE   = 1.58
MIN_FITNESS  = 1.0
MAX_TURNOVER = 0.70

# discrete options grid
DECAY_OPTIONS  = [0, 5, 10, 20, 60]
NEUT_OPTIONS   = ['INDUSTRY', 'SUBINDUSTRY', 'MARKET', 'CROWDING', 'STATISTICAL']
WINDOW_OPTIONS = [5, 10, 20, 40, 60, 120, 252]

In [ ]:
_type_frames = []
for _dtype in ['MATRIX', 'VECTOR', 'GROUP']:
    _df = ace.get_datafields(
        s,
        instrument_type='EQUITY',
        region=REGION,
        delay=DELAY,
        universe=UNIVERSE,
        data_type=_dtype,
        dataset_id="model109"
    )
    if not _df.empty:
        _df['type'] = _dtype
        _type_frames.append(_df)

df_fields = pd.concat(_type_frames, ignore_index=True) if _type_frames else pd.DataFrame()

print(f'Total fields fetched: {len(df_fields)}')
if not df_fields.empty:
    print(df_fields['type'].value_counts())

## 3. Field pools

Split the fetched datafields into typed pools so the template engine and mutation operators know which fields are scalar-compatible (`MATRIX`), need a `vec_*` wrapper (`VECTOR`), or are valid `{G}` group expressions (`GROUP`).

In [ ]:
MATRIX_FIELDS = df_fields.loc[df_fields['type'] == 'MATRIX', 'id'].tolist()
VECTOR_FIELDS = df_fields.loc[df_fields['type'] == 'VECTOR', 'id'].tolist()
GROUP_FIELDS  = df_fields.loc[df_fields['type'] == 'GROUP', 'id'].tolist()

field_type_map = {}
field_type_map.update({f: 'MATRIX' for f in MATRIX_FIELDS})
field_type_map.update({f: 'VECTOR' for f in VECTOR_FIELDS})
field_type_map.update({f: 'GROUP'  for f in GROUP_FIELDS})

VEC_WRAPPERS = ['vec_avg', 'vec_sum', 'vec_min', 'vec_max', 'vec_stddev', 'vec_ir']

GROUP_OPTIONS = GROUP_FIELDS + [
    'market',
    'sector',
    'industry',
    'subindustry',
    'densify(country)',
    'group_cartesian_product(country, subindustry)',
    'group_cartesian_product(country, sector)',
]

print(f'MATRIX fields: {len(MATRIX_FIELDS)}')
print(f'VECTOR fields: {len(VECTOR_FIELDS)}')
print(f'GROUP fields:  {len(GROUP_FIELDS)}')
print(f'Group options: {len(GROUP_OPTIONS)}')

## 4. Expression template library

Each template is a `(name, pattern)` pair. Patterns use `{F}` / `{F2}` for scalar-compatible fields (a `MATRIX` field directly, or a `VECTOR` field wrapped in `vec_*`), `{W}` / `{W2}` for lookback windows, and `{G}` for a group expression.

In [ ]:
# template library
# {F}  = a scalar-compatible field (MATRIX directly, or vec_*(VECTOR))
# {F2} = second independent scalar field
# {W}  = lookback window
# {W2} = second window
# {G}  = group expression string
TEMPLATES = [
    ('seed_alpha',    'ts_rank({F}, {W})'),
    ('ts_rank_cs',    'zscore(ts_rank({F}, {W}))'),
    ('ts_zscore_cs',  'rank(ts_zscore({F}, {W}))'),
    ('ts_mean_cs',    'zscore(ts_mean({F}, {W}))'),
    ('ts_delta_cs',   'zscore(ts_delta({F}, {W}))'),
    ('ts_std_cs',     'rank(inverse(ts_std_dev({F}, {W})))'),

    ('bf_rank',       'zscore(ts_rank(ts_backfill({F}, {W}), {W2}))'),
    ('bf_zscore',     'rank(ts_zscore(ts_backfill({F}, {W}), {W2}))'),

    ('grp_neut_rank',   'group_neutralize(zscore(ts_rank({F}, {W})), {G})'),
    ('grp_neut_zscore', 'group_neutralize(rank(ts_zscore({F}, {W})), {G})'),
    ('grp_rank',        'group_rank(ts_rank({F}, {W}), {G})'),
    ('grp_zscore',       'group_zscore(ts_zscore({F}, {W}), {G})'),

    ('diff_rank',     'zscore(subtract(ts_mean({F}, {W}), ts_mean({F2}, {W2})))'),
    ('ratio_rank',    'zscore(divide(ts_mean({F}, {W}), ts_mean({F2}, {W2})))'),
    ('corr_cs',       'zscore(ts_corr({F}, {F2}, {W}))'),
    ('diff_delta',    'zscore(subtract(ts_delta({F}, {W}), ts_delta({F2}, {W2})))'),

    ('tanh_ts',       'tanh(zscore(ts_rank({F}, {W})))'),
    ('sigmoid_ts',    'sigmoid(rank(ts_delta({F}, {W})))'),

    ('grp_diff',      'group_neutralize(zscore(subtract(ts_mean({F}, {W}), ts_mean({F2}, {W2}))), {G})'),
    ('grp_corr',      'group_neutralize(zscore(ts_corr({F}, {F2}, {W})), {G})'),
]

print(f'Template library: {len(TEMPLATES)} patterns')

## 5. Individual representation and creation

An individual is represented as a single string: the rendered FASTEXPR expression. DEAP's `creator` wraps that string with a `FitnessMax`-style fitness attribute. `make_field_term()` decides whether a field needs a `vec_*` wrapper; `render_template()` fills in a template's placeholders with randomly sampled fields/windows/groups; `random_individual()` picks a random template and renders it.

In [ ]:
creator.create('FitnessMax', base.Fitness, weights=(1.0,))
creator.create('Individual', str, fitness=creator.FitnessMax)


def make_field_term(field: str) -> str:
    """Wrap a VECTOR field in a random vec_* operator; pass MATRIX fields through."""
    if field_type_map.get(field) == 'VECTOR':
        wrapper = random.choice(VEC_WRAPPERS)
        return f'{wrapper}({field})'
    return field


def render_template(name: str, pattern: str) -> str:
    """Fill in a template's {F}/{F2}/{W}/{W2}/{G} placeholders with random choices."""
    all_fields = MATRIX_FIELDS + VECTOR_FIELDS
    expr = pattern
    if '{F}' in expr:
        expr = expr.replace('{F}', make_field_term(random.choice(all_fields)))
    if '{F2}' in expr:
        expr = expr.replace('{F2}', make_field_term(random.choice(all_fields)))
    if '{W}' in expr:
        expr = expr.replace('{W}', str(random.choice(WINDOW_OPTIONS)))
    if '{W2}' in expr:
        expr = expr.replace('{W2}', str(random.choice(WINDOW_OPTIONS)))
    if '{G}' in expr:
        expr = expr.replace('{G}', random.choice(GROUP_OPTIONS))
    return expr


def random_individual() -> 'creator.Individual':
    name, pattern = random.choice(TEMPLATES)
    expr = render_template(name, pattern)
    return creator.Individual(expr)


print('Sample individuals:')
for _ in range(3):
    print(' ', random_individual())

## 6. Fitness evaluation

Batches individuals through `ace.simulate_multi_alpha`, pulls back IS Sharpe / fitness / turnover, and combines them into a single composite score. Individuals that fail the hard thresholds (`MIN_SHARPE`, `MIN_FITNESS`, `MAX_TURNOVER`) get an infeasible score (`-100`) so they lose every tournament without crashing downstream statistics.

In [ ]:
INFEASIBLE_SCORE = -100.0


def composite_score(sharpe: float, fitness: float, turnover: float) -> float:
    """Composite fitness: reward Sharpe and fitness, lightly penalise high turnover."""
    if sharpe < MIN_SHARPE or fitness < MIN_FITNESS or turnover > MAX_TURNOVER:
        return INFEASIBLE_SCORE
    turnover_penalty = max(0.0, turnover - 0.3) * 0.5
    return float(sharpe + fitness - turnover_penalty)


def build_simulate_data(expr: str) -> dict:
    decay = random.choice(DECAY_OPTIONS)
    neutralization = random.choice(NEUT_OPTIONS)
    settings = dict(SIM_SETTINGS)
    settings['decay'] = decay
    settings['neutralization'] = neutralization
    return {
        'type': 'REGULAR',
        'settings': settings,
        'regular': expr,
    }


def evaluate_population(individuals: list) -> None:
    """Batch-simulate individuals lacking a valid fitness, in chunks of BATCH_SIZE."""
    pending = [ind for ind in individuals if not ind.fitness.valid]
    for batch_start in range(0, len(pending), BATCH_SIZE):
        batch = pending[batch_start:batch_start + BATCH_SIZE]
        alpha_list = [build_simulate_data(str(ind)) for ind in batch]
        results = ace.simulate_multi_alpha(s, alpha_list)

        for ind, result in zip(batch, results):
            alpha_id = result.get('alpha_id') if result else None
            if not alpha_id:
                ind.fitness.values = (INFEASIBLE_SCORE,)
                continue
            stats = ace.get_specified_alpha_stats(s, alpha_id)
            sharpe = stats.get('sharpe', 0.0)
            fitness = stats.get('fitness', 0.0)
            turnover = stats.get('turnover', 1.0)
            ind.fitness.values = (composite_score(sharpe, fitness, turnover),)

## 7. Genetic operators

### Crossover

Two crossover modes, each applied with independent probability so a single mating event can trigger either, both, or neither:

- **Field swap**: each child donates one of its field tokens to the other
- **Window perturb**: each numerical window in both expressions is nudged by ±20–50 %

### Mutation

Four mutation modes:

- **Field swap**: replace one field with a random compatible alternative
- **Operator swap**: replace a time-series operator (e.g. `ts_rank`) with a semantically similar one (e.g. `ts_zscore`)
- **Window perturb**: nudge a randomly chosen numeric window
- **Sign flip**: negate the entire expression (adds `-(...)` wrapper or removes an existing one)

In [ ]:
_WINDOW_RE = re.compile(r'(?<![\w.])(\d+)(?![\w.])')


def _perturb_windows_in_expr(expr: str) -> str:
    """Nudge every bare integer window token in expr by +/-20-50%."""
    def _bump(match: 're.Match') -> str:
        old_w = int(match.group(1))
        if old_w not in WINDOW_OPTIONS and old_w < 2:
            return match.group(0)
        pct = random.uniform(0.2, 0.5) * random.choice([-1, 1])
        new_w = max(1, int(round(old_w * (1 + pct))))
        return str(new_w)
    return _WINDOW_RE.sub(_bump, expr)


def _swap_field_tokens(expr1: str, expr2: str) -> tuple:
    """Donate one field token from expr2 into expr1 and vice versa."""
    all_fields = MATRIX_FIELDS + VECTOR_FIELDS
    found1 = [f for f in all_fields if f in expr1]
    found2 = [f for f in all_fields if f in expr2]
    if not found1 or not found2:
        return expr1, expr2
    f1 = random.choice(found1)
    f2 = random.choice(found2)
    new_expr1 = expr1.replace(f1, f2, 1)
    new_expr2 = expr2.replace(f2, f1, 1)
    return new_expr1, new_expr2


def cx_alpha(ind1: 'creator.Individual', ind2: 'creator.Individual') -> tuple:
    """Crossover: field swap and/or window perturb, each with its own coin flip.

    Individuals are plain (immutable) strings under the hood, so this returns
    brand-new Individual objects rather than mutating ind1/ind2 in place —
    callers (DEAP's toolbox.mate) use the returned tuple.
    """
    expr1, expr2 = str(ind1), str(ind2)

    if random.random() < 0.5:
        expr1, expr2 = _swap_field_tokens(expr1, expr2)

    if random.random() < 0.5:
        expr1 = _perturb_windows_in_expr(expr1)
        expr2 = _perturb_windows_in_expr(expr2)

    new_ind1 = creator.Individual(expr1)
    new_ind2 = creator.Individual(expr2)
    return new_ind1, new_ind2

In [ ]:
_SWAP_GROUPS = [
    ['ts_rank', 'ts_zscore', 'ts_scale', 'ts_quantile'],
    ['ts_mean', 'ts_sum', 'ts_median'],
    ['ts_delta', 'ts_av_diff', 'ts_min_diff', 'ts_max_diff'],
    ['ts_min', 'ts_max'],
    ['ts_std_dev', 'ts_skewness', 'ts_entropy'],
    ['ts_decay_linear', 'ts_backfill'],
    ['rank', 'zscore', 'scale', 'normalize', 'quantile'],
    ['group_rank', 'group_zscore', 'group_neutralize', 'group_scale'],
    ['vec_avg', 'vec_sum', 'vec_min', 'vec_max', 'vec_stddev'],
    ['subtract', 'add'],
    ['divide', 'multiply'],
    ['tanh', 'sigmoid'],
]


def _swap_field_in_expr(expr: str, field_type_map: dict) -> str:
    """Replace one field token in expr with a randomly chosen compatible one"""
    all_fields = MATRIX_FIELDS + VECTOR_FIELDS
    found = [f for f in all_fields if f in expr]
    if not found:
        return expr
    old_f = random.choice(found)
    new_f = random.choice(all_fields)
    if new_f == old_f:
        return expr

    old_type = field_type_map.get(old_f, 'MATRIX')
    new_type = field_type_map.get(new_f, 'MATRIX')

    # If we are swapping a MATRIX for a VECTOR (or vice versa), need to add / remove the vec_* wrapper accordingly
    if old_type == 'VECTOR' and new_type == 'MATRIX':
        for wrapper in VEC_WRAPPERS:
            wrapped_old = f'{wrapper}({old_f})'
            if wrapped_old in expr:
                return expr.replace(wrapped_old, new_f, 1)
        return expr.replace(old_f, new_f, 1)

    if old_type == 'MATRIX' and new_type == 'VECTOR':
        wrapped_new = make_field_term(new_f)
        return expr.replace(old_f, wrapped_new, 1)

    return expr.replace(old_f, new_f, 1)


def _swap_operator_in_expr(expr: str) -> str:
    """Replace one operator token with a semantically similar one from _SWAP_GROUPS"""
    random.shuffle(_SWAP_GROUPS)
    for group in _SWAP_GROUPS:
        found = [op for op in group if re.search(rf'\b{re.escape(op)}\(', expr)]
        if found:
            old_op = random.choice(found)
            choices = [op for op in group if op != old_op]
            if not choices:
                continue
            new_op = random.choice(choices)
            return re.sub(rf'\b{re.escape(old_op)}\(', f'{new_op}(', expr, count=1)
    return expr


def _flip_sign_in_expr(expr: str) -> str:
    """Negate the entire expression, or remove an existing top-level negation"""
    stripped = expr.strip()
    if stripped.startswith('-(') and stripped.endswith(')'):
        return stripped[2:-1]
    return f'-({stripped})'


def mut_alpha(individual: 'creator.Individual') -> tuple:
    """Mutation: pick one of the four mutation modes and apply it.

    Returns a 1-tuple containing a brand-new Individual (str subclasses are
    immutable, so this can't mutate `individual` in place). DEAP's
    toolbox.mutate expects exactly this (individual,) return shape.
    """
    expr = str(individual)
    mode = random.choice(['field_swap', 'operator_swap', 'window_perturb', 'sign_flip'])

    if mode == 'field_swap':
        expr = _swap_field_in_expr(expr, field_type_map)
    elif mode == 'operator_swap':
        expr = _swap_operator_in_expr(expr)
    elif mode == 'window_perturb':
        expr = _perturb_windows_in_expr(expr)
    elif mode == 'sign_flip':
        expr = _flip_sign_in_expr(expr)

    return (creator.Individual(expr),)

## 8. Toolbox, population, and Hall of Fame

Registers the DEAP toolbox (`select` = tournament, `mate` = `cx_alpha`, `mutate` = `mut_alpha`), seeds the initial population with `random_individual()`, evaluates it once up front, and sets up the Hall of Fame plus a Statistics/Logbook pair for tracking convergence.

In [ ]:
toolbox = base.Toolbox()
toolbox.register('individual', random_individual)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)

toolbox.register('select', tools.selTournament, tournsize=TOURNAMENT_SIZE)
toolbox.register('mate', cx_alpha)
toolbox.register('mutate', mut_alpha)

population = toolbox.population(n=POPULATION_SIZE)

print(f'Seeding initial population of {len(population)} individuals...')
evaluate_population(population)

hof = tools.HallOfFame(HOF_SIZE)
hof.update(population)

logbook = tools.Logbook()
logbook.header = ['gen', 'n_evals', 'avg', 'std', 'min', 'max']

_init_valid_fits = [ind.fitness.values[0] for ind in population if ind.fitness.valid and ind.fitness.values[0] > -100]
logbook.record(
    gen=0,
    n_evals=len(population),
    avg=float(np.mean(_init_valid_fits)) if _init_valid_fits else float('nan'),
    std=float(np.std(_init_valid_fits)) if _init_valid_fits else float('nan'),
    min=float(np.min(_init_valid_fits)) if _init_valid_fits else float('nan'),
    max=float(np.max(_init_valid_fits)) if _init_valid_fits else float('nan'),
)
print(logbook.stream)

## 9. Evolutionary loop

In [ ]:
for gen in range(1, N_GENERATIONS + 1):
    print(f'\n--- Generation {gen} / {N_GENERATIONS} ---')

    # Step 1 — tournament selection (with replacement to fill population)
    offspring = toolbox.select(population, len(population))
    offspring = list(map(toolbox.clone, offspring))

    # Step 2 — crossover
    # NOTE: individuals here are plain (immutable) strings, so mate/mutate
    # return brand-new Individuals rather than updating in place — we have
    # to write the results back into `offspring` ourselves.
    for i in range(0, len(offspring) - 1, 2):
        if random.random() < CXPB:
            child1, child2 = toolbox.mate(offspring[i], offspring[i + 1])
            offspring[i], offspring[i + 1] = child1, child2

    # Step 3 — mutation
    for i, mutant in enumerate(offspring):
        if random.random() < MUTPB:
            (offspring[i],) = toolbox.mutate(mutant)

    # Step 4 — evaluate individuals
    stale = [ind for ind in offspring if not ind.fitness.valid]
    print(f'  evaluating {len(stale)} individuals...')
    evaluate_population(offspring)

    # Step 5 — replace population
    population[:] = offspring

    # Step 6 — recording
    hof.update(population)
    valid_fits = [
        ind.fitness.values[0]
        for ind in population
        if ind.fitness.valid and ind.fitness.values[0] > -100
    ]
    record = {
        'gen': gen,
        'n_evals': len(stale),
        'avg': float(np.mean(valid_fits)) if valid_fits else float('nan'),
        'std': float(np.std(valid_fits)) if valid_fits else float('nan'),
        'min': float(np.min(valid_fits)) if valid_fits else float('nan'),
        'max': float(np.max(valid_fits)) if valid_fits else float('nan'),
    }
    logbook.record(**record)
    print(logbook.stream)

print('\nGA complete.')

## 10. Results

### 10.1 Generation statistics

- Logbook records score statistics of feasible sub-population each generation
- A rising mean + narrowing standard deviation = population is converging

In [ ]:
log_df = pd.DataFrame(logbook)
display(log_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

gens = log_df['gen']
avgs = log_df['avg']
maxes = log_df['max']
stds = log_df['std']

ax = axes[0]
ax.plot(gens, avgs,  label='Mean score', linewidth=2)
ax.plot(gens, maxes, label='Max score',  linewidth=2, linestyle='--')
ax.fill_between(gens, avgs - stds, avgs + stds, alpha=0.2, label='±1 std')
ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
ax.set_xlabel('Generation')
ax.set_ylabel('Composite score')
ax.set_title('Population fitness over generations')
ax.legend()

ax = axes[1]
ax.plot(gens, stds, color='orange', linewidth=2)
ax.set_xlabel('Generation')
ax.set_ylabel('Std dev of score')
ax.set_title('Score diversity over generations')

plt.tight_layout()
plt.show()

### 10.2 Hall of Fame

The best individuals found across the entire run, regardless of which generation they appeared in.

In [ ]:
hof_df = pd.DataFrame({
    'rank': range(1, len(hof) + 1),
    'expression': [str(ind) for ind in hof],
    'score': [ind.fitness.values[0] for ind in hof],
})
display(hof_df)